In [1]:
import numpy as np
import os
import pandas as pd
import glob
from brits_tensorflow.model import BRITS
from brits_tensorflow.plots import plot
import pickle
import json
import random

In [ ]:
# 指定包含 .csv.gz 檔案的根目錄路徑
root_directory = "C:/yu/physionet.org/files/mimic4wdb/0.1.0/waves/"

# 存儲每個 CSV 檔案的數據
data_frames = {}

# 找到所有 .csv.gz 檔案
for folder, subfolders, files in os.walk(root_directory):
    for file in files:
        if file.endswith(".csv.gz"):
            # 構建完整的檔案路徑
            file_path = os.path.join(folder, file)
            # 使用 pandas 的 read_csv 函式讀取 gzip 壓縮的 CSV 檔案，並存儲在字典中
            data_frames[file] = pd.read_csv(file_path, compression='gzip')

for key, df in data_frames.items():
    df = (
        df.dropna(axis=1, thresh=0.1*len(df))  # axis=1 表示按列操作，thresh 是非缺失值的最小数量，如果超過這個數量則保留
          .assign(Group=(df['time'] - df['time'][0]) // 86400)  # 計算每个數據點   #合併每3600個timestep
          .groupby('Group').apply(lambda x: x.mean(skipna=True))  # 使用 groupby() 方法按 'Group' 列分组，計算每组的平均值
          .drop(columns=['time', 'Group'])
    )

    # 重新整理索引
    new_index = list(range(df.index[-1] + 1))
    data_frames[key] = df.reindex(new_index)  # 放回字典

with open('data_frames.pkl', 'wb') as file:
    pickle.dump(data_frames, file)

In [2]:
# 將資料匯入(從這裡開始跑程式就可以)
with open('result/data_frames.pkl', 'rb') as file:
    data_frames = pickle.load(file)

In [3]:
# 幫資料刪除最後一筆，並且至少大於100筆
for key in data_frames:
    data_frames[key] = data_frames[key].iloc[:-1]

data_frames = {key: value for key, value in data_frames.items() if len(value) >= 100}

In [4]:
# model

def models(_evals, _values):
    model = BRITS(x=_evals, units=100, timesteps=1)

    model.fit(
        learning_rate=0.001,
        batch_size=16,
        epochs=100,
        verbose=True
    )

    x_hat = model.impute(x=_values)

    return x_hat


# 將數據np.array拉成一直線

def getvalue(array):
    x = array.reshape(-1)

    # 使用布爾值作為索引，抓取對應位置的值
    x = x[eval_value]

    return x


# 將資料變回原來的格式
def resize(_x_hat, _values):

    all_indexes = set(range(df.shape[0]))
    # NumPy轉回DataFrame，索引不變
    _x_hat = pd.DataFrame(_x_hat.to_numpy(), columns=_values.columns, index=_values.index)
    # 找出缺失索引
    missing_indexes = list(all_indexes - set(_x_hat.index))
    # 使用 reindex 方法填充遺失索引
    _x_hat = _x_hat.reindex(_x_hat.index.union(missing_indexes))

    return _x_hat

原始BRITS

In [ ]:
x_hat1_dict = {}

for key, df in data_frames.items():
    # 原始

    # 資料前處理 

    #轉成 NumPy
    matrix = df.to_numpy()
    #矩陣維度
    shp = matrix.shape
    #變成1維數組
    evals = matrix.reshape(-1)

    #有值的位置
    indices = np.where(~np.isnan(evals))[0].tolist()
    #挖空了有值的10%，replace=False為不重複挖
    #這個就是那10%的資料
    random.seed(1234)
    indices = np.random.choice(indices, len(indices) // 10, replace=False)

    #複製df
    values = evals.copy()
    #把一部分有值的結果用NAN替代，用於驗證
    values[indices] = np.nan

    #比較values與evals的不同處，true為不同的地方
    eval_value = (~np.isnan(values)) ^ (~np.isnan(evals))

    #還原的矩陣
    #原始數據
    evals = evals.reshape(shp)
    #增加10%遺失數據
    values = values.reshape(shp)

    # model
    x_hat = models(evals, values)

    if key not in x_hat1_dict:
        x_hat1_dict[key] = {}

    x_hat1_dict[key]['0'] = pd.DataFrame(x_hat)

In [ ]:
# 存檔
with open('x_hat1_dict.pkl', 'wb') as file:
    pickle.dump(x_hat1_dict, file)

M-BRITS

In [87]:
# 看現在步數間格多少 (2 3 4 5 ....)
num_slices = 2

In [ ]:
x_dict = {}

for key, df in data_frames.items():
    # 原始

    # 資料前處理 

    #轉成 NumPy
    matrix = df.to_numpy()
    #矩陣維度
    shp = matrix.shape
    #變成1維數組
    evals = matrix.reshape(-1)

    #有值的位置
    indices = np.where(~np.isnan(evals))[0].tolist()
    #挖空了有值的10%，replace=False為不重複挖
    #這個就是那10%的資料
    random.seed(1234)
    indices = np.random.choice(indices, len(indices) // 10, replace=False)

    #複製df
    values = evals.copy()
    #把一部分有值的結果用NAN替代，用於驗證
    values[indices] = np.nan

    #比較values與evals的不同處，true為不同的地方
    eval_value = (~np.isnan(values)) ^ (~np.isnan(evals))

    #還原的矩陣
    #原始數據
    evals = evals.reshape(shp)
    #增加10%遺失數據
    values = values.reshape(shp)

    evals_slices = [pd.DataFrame(evals).iloc[slice(i, None, num_slices)] for i in range(num_slices)]
    values_slices = [pd.DataFrame(values).iloc[slice(i, None, num_slices)] for i in range(num_slices)]

    bigger_than_two_x_hat = {}

    for i in range(num_slices):
        evals_slices[i]
        values_slices[i]

        result = models(evals_slices[i].values, values_slices[i].values)
        bigger_than_two_x_hat[f'{i}'] = pd.DataFrame(result)

        if key not in x_dict:
            x_dict[key] = {}

        x_dict[key][f'{i}'] = resize(bigger_than_two_x_hat[f'{i}'], values_slices[i])


In [89]:
# 存檔
with open(f'result/x_hat{num_slices}_dict.pkl', 'wb') as file:
    pickle.dump(x_dict, file) # 手動更改要存的檔案名稱

In [24]:
# # 讀取pickle文件
# with open('result/x_hat1_dict.pkl', 'rb') as file:
#     x_hat1_dict = pickle.load(file)